# Clinical-Grade CXR Report Generation: Fine-Tuning PaliGemma on MIMIC-CXR

This notebook builds an end-to-end multimodal training pipeline for chest X-ray report generation using **QLoRA** on **google/paligemma-3b-pt-224**.

Clinical framing:
- We explicitly parse and train on **Findings** and **Impression** because they represent the most clinically actionable and standardized report components.
- We use quantization + LoRA to reduce memory footprint, enabling training on commodity GPUs while preserving adaptation quality.

Assumptions:
- You have access to MIMIC-CXR-JPG files and metadata/report CSVs.
- You run on a GPU notebook with at least ~16GB VRAM.

## Phase 1: Environment & MIMIC-CXR Data Orchestration

In [ ]:
# Code Cell 1: Install required libraries
# In Kaggle/Colab, run once per fresh runtime.

!pip -q install -U bitsandbytes peft accelerate transformers datasets evaluate nltk rouge-score sentencepiece

In [ ]:
# Code Cell 2: Kaggle MIMIC-CXR dataset loader + report section extraction

import ast
import os
import re
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from PIL import Image

# -----------------------------
# Kaggle dataset layout
# -----------------------------
# The Kaggle mirror used by the sample notebook stores the cleaned study-level CSVs at:
#   /kaggle/input/datasets/simhadrisadaram/mimic-cxr-dataset/mimic_cxr_aug_train.csv
#   /kaggle/input/datasets/simhadrisadaram/mimic-cxr-dataset/mimic_cxr_aug_validate.csv
# and the JPG hierarchy under:
#   /kaggle/input/datasets/simhadrisadaram/mimic-cxr-dataset/official_data_iccv_final/files
# which contains paths like: files/p10/p10000032/s50414267/02aa804e-....jpg

DATASET_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/simhadrisadaram/mimic-cxr-dataset"),
    Path("/kaggle/input/simhadrisadaram/mimic-cxr-dataset"),
    Path("/kaggle/input/mimic-cxr-dataset"),
]

CLEANED_CSV_NAMES = ["mimic_cxr_aug_train.csv", "mimic_cxr_aug_validate.csv"]
RAW_METADATA_NAME = "mimic-cxr-2.0.0-metadata.csv.gz"
RAW_REPORTS_NAME = "mimic-cxr-2.0.0-reports.csv.gz"
FILES_SUBDIRS = [Path("official_data_iccv_final/files"), Path("files")]

SUBSET_SIZE = 2000
SEED = 42
np.random.seed(SEED)


def _first_existing_path(candidates: List[Path]) -> Optional[Path]:
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


def _discover_dataset_root() -> Optional[Path]:
    for root in DATASET_ROOT_CANDIDATES:
        if root.exists():
            return root
    return None


def parse_maybe_list(value) -> List[str]:
    """
    Parse values that may be stored as native lists, JSON-like strings, or plain scalars.

    The Kaggle mirror used in the sample notebook stores multi-view paths and text fields
    in list-like columns, so we normalize them here before building study-level records.
    """
    if value is None:
        return []
    if isinstance(value, float) and np.isnan(value):
        return []
    if isinstance(value, list):
        return [str(x).strip() for x in value if str(x).strip()]
    if isinstance(value, tuple):
        return [str(x).strip() for x in value if str(x).strip()]
    if not isinstance(value, str):
        return [str(value).strip()] if str(value).strip() else []

    text = value.strip()
    if not text:
        return []

    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            return [str(x).strip() for x in parsed if str(x).strip()]
        if isinstance(parsed, tuple):
            return [str(x).strip() for x in parsed if str(x).strip()]
    except Exception:
        pass

    if text.startswith("[") and text.endswith("]"):
        # Last-resort cleanup for stringified lists with non-literal formatting.
        parts = [part.strip().strip("'\"") for part in text[1:-1].split(",")]
        return [part for part in parts if part]

    return [text]


def extract_study_id(image_path: str) -> Optional[str]:
    if image_path is None:
        return None
    match = re.search(r"/s(\d+)/", str(image_path).replace("\\", "/"))
    return match.group(1) if match else None


def resolve_image_path(rel_path: str, image_root: Path) -> Optional[Path]:
    """
    Resolve a relative or partially rooted MIMIC-CXR JPG path.

    The Kaggle mirror often stores paths relative to the files/ subtree.
    We accept several variants so notebook execution stays robust across mounts.
    """
    if rel_path is None:
        return None

    path_str = str(rel_path).strip().replace("\\", "/")
    if not path_str:
        return None

    candidate = Path(path_str)
    if candidate.exists():
        return candidate

    if path_str.startswith("files/"):
        path_str = path_str[len("files/"):]

    if path_str.startswith("/files/"):
        path_str = path_str[len("/files/"):]

    candidate = image_root / path_str.lstrip("/")
    if candidate.exists():
        return candidate

    return None


def choose_view_paths_from_lists(pa_items: List[str], ap_items: List[str], lat_items: List[str], image_root: Path) -> List[Path]:
    """
    Prefer frontal views first, matching the processing strategy shown in the sample notebook.

    Clinical rationale:
    - PA/AP views are the primary source for report drafting.
    - Lateral views are a fallback when frontal images are unavailable.
    """
    pa_paths = [p for p in (resolve_image_path(x, image_root) for x in pa_items) if p is not None]
    ap_paths = [p for p in (resolve_image_path(x, image_root) for x in ap_items) if p is not None]
    lat_paths = [p for p in (resolve_image_path(x, image_root) for x in lat_items) if p is not None]

    frontal = pa_paths + ap_paths
    if frontal:
        return frontal
    return lat_paths


def extract_findings_impression(report_text: str) -> Tuple[str, str]:
    """
    Extract Findings and Impression from a free-text report.

    Why this is the right target:
    - Findings captures visible imaging evidence.
    - Impression captures the clinically concise diagnostic summary.
    - Training on both improves report fidelity and downstream evaluation.
    """
    if not isinstance(report_text, str) or not report_text.strip():
        return "", ""

    text = report_text.strip()
    findings_match = re.search(
        r"(?is)\bfindings\b\s*:\s*(.*?)(?=\n\s*\bimpression\b\s*:|\n\s*[A-Z][A-Z\s]{2,}:|\Z)",
        text,
    )
    impression_match = re.search(
        r"(?is)\bimpression\b\s*:\s*(.*?)(?=\n\s*[A-Z][A-Z\s]{2,}:|\Z)",
        text,
    )

    findings = re.sub(r"\s+", " ", findings_match.group(1)).strip() if findings_match else ""
    impression = re.sub(r"\s+", " ", impression_match.group(1)).strip() if impression_match else ""

    if not findings and not impression:
        normalized = re.sub(r"\s+", " ", text).strip()
        parts = re.split(r"(?i)\bimpression\b\s*:\s*", normalized, maxsplit=1)
        if len(parts) == 2:
            findings = parts[0].strip(" :")
            impression = parts[1].strip(" :")
        else:
            findings = normalized[:1200]
            impression = normalized[:400]

    return findings, impression


def build_structured_report(raw_report_text: str) -> str:
    findings, impression = extract_findings_impression(raw_report_text)
    if findings or impression:
        return f"Findings: {findings}\nImpression: {impression}".strip()
    compact = re.sub(r"\s+", " ", raw_report_text or "").strip()
    return f"Findings: {compact[:1200]}\nImpression: {compact[:400]}".strip()


def build_study_records_from_row(row: pd.Series, text_col: str, image_root: Path) -> List[Dict]:
    """
    Convert a source row into one or more study-level records.

    The sample notebook uses a study-level expansion strategy because a single subject row can
    contain multiple studies and multiple view paths. We preserve that structure here so the
    training notebook aligns with the Kaggle mirror file layout.
    """
    image_items = parse_maybe_list(row.get("image"))
    text_items = parse_maybe_list(row.get(text_col))
    pa_items = parse_maybe_list(row.get("PA"))
    ap_items = parse_maybe_list(row.get("AP"))
    lat_items = parse_maybe_list(row.get("Lateral"))

    ordered_study_ids: List[str] = []
    for image_path in image_items:
        study_id = extract_study_id(image_path)
        if study_id and study_id not in ordered_study_ids:
            ordered_study_ids.append(study_id)

    if not ordered_study_ids and row.get("study_id") is not None:
        ordered_study_ids = [str(row.get("study_id"))]

    if not text_items and isinstance(row.get(text_col), str):
        text_items = [row.get(text_col)]

    if not ordered_study_ids or not text_items:
        return []

    if len(text_items) == 1 and len(ordered_study_ids) > 1:
        text_items = text_items * len(ordered_study_ids)

    records = []
    for report_text_raw, study_id in zip(text_items, ordered_study_ids):
        study_pa = [x for x in pa_items if extract_study_id(x) == study_id]
        study_ap = [x for x in ap_items if extract_study_id(x) == study_id]
        study_lat = [x for x in lat_items if extract_study_id(x) == study_id]

        study_image_paths = choose_view_paths_from_lists(study_pa, study_ap, study_lat, image_root)

        # Some rows may carry a direct image path instead of the view-list columns.
        if not study_image_paths:
            direct_candidates = []
            for key in ["image_path", "path", "filepath", "jpg_path", "dicom_path"]:
                if key in row.index and pd.notna(row.get(key)):
                    direct_candidates.append(resolve_image_path(str(row.get(key)), image_root))
            study_image_paths = [p for p in direct_candidates if p is not None]

        if not study_image_paths:
            continue

        structured_report = build_structured_report(report_text_raw)
        findings, impression = extract_findings_impression(structured_report)

        records.append({
            "subject_id": row.get("subject_id"),
            "study_id": study_id,
            "report_text": structured_report,
            "findings": findings,
            "impression": impression,
            "image_paths": [str(p) for p in study_image_paths],
            "image_path": str(study_image_paths[0]),
            "num_views_used": len(study_image_paths),
            "raw_report_text": report_text_raw,
            "source_row_index": row.name,
        })

    return records


# -----------------------------
# Load cleaned Kaggle CSVs first, with raw MIMIC CSV fallback
# -----------------------------
DATA_ROOT = _discover_dataset_root()
assert DATA_ROOT is not None, (
    "Could not find the Kaggle MIMIC-CXR dataset root. "
    "Expected one of the mounted input directories from the sample notebook."
)

IMAGE_ROOT = _first_existing_path([DATA_ROOT / subdir for subdir in FILES_SUBDIRS])
assert IMAGE_ROOT is not None, f"Could not find image root under: {DATA_ROOT}"

cleaned_csv_paths = [DATA_ROOT / name for name in CLEANED_CSV_NAMES if (DATA_ROOT / name).exists()]
raw_metadata_path = DATA_ROOT / RAW_METADATA_NAME
raw_reports_path = DATA_ROOT / RAW_REPORTS_NAME

print(f"Detected dataset root: {DATA_ROOT}")
print(f"Detected image root:   {IMAGE_ROOT}")
print(f"Cleaned CSVs found:     {len(cleaned_csv_paths)}")
for path in cleaned_csv_paths:
    print(f" - {path}")

text_column_candidates = ["text", "text_list", "report", "report_text", "findings"]

study_records: List[Dict] = []

if cleaned_csv_paths:
    cleaned_frames = [pd.read_csv(path) for path in cleaned_csv_paths]
    source_df = pd.concat(cleaned_frames, ignore_index=True)
    print(f"Loaded cleaned Kaggle rows: {len(source_df):,}")

    text_col = next((c for c in text_column_candidates if c in source_df.columns), None)
    if text_col is None:
        raise ValueError(
            f"Could not find a usable text column in cleaned CSVs. Available columns: {list(source_df.columns)}"
        )

    print(f"Using cleaned text column: {text_col}")
    for _, row in source_df.iterrows():
        study_records.extend(build_study_records_from_row(row, text_col=text_col, image_root=IMAGE_ROOT))

else:
    assert raw_metadata_path.exists(), f"Missing raw metadata file: {raw_metadata_path}"
    assert raw_reports_path.exists(), f"Missing raw reports file: {raw_reports_path}"

    # Fallback path if the Kaggle mirror only exposes the original gzip pair.
    meta_df = pd.read_csv(raw_metadata_path, compression="gzip")
    rep_df = pd.read_csv(raw_reports_path, compression="gzip")

    merged = meta_df.merge(rep_df[["subject_id", "study_id", "report"]], on=["subject_id", "study_id"], how="inner")
    merged[["findings", "impression"]] = merged["report"].apply(lambda x: pd.Series(extract_findings_impression(x)))
    merged = merged[(merged["findings"].str.len() > 0) | (merged["impression"].str.len() > 0)].copy()
    merged["image_path"] = merged.apply(
        lambda r: str(IMAGE_ROOT / f"p{str(int(r['subject_id']))[:3]}" / f"p{int(r['subject_id'])}" / f"s{int(r['study_id'])}" / f"{r['dicom_id']}.jpg"),
        axis=1,
    )
    merged["exists"] = merged["image_path"].map(os.path.exists)
    merged = merged[merged["exists"]].copy()
    merged["report_text"] = merged.apply(
        lambda r: f"Findings: {r['findings'].strip()}\nImpression: {r['impression'].strip()}".strip(), axis=1
    )
    source_df = merged
    study_records = [
        {
            "subject_id": r["subject_id"],
            "study_id": str(r["study_id"]),
            "report_text": r["report_text"],
            "findings": r["findings"],
            "impression": r["impression"],
            "image_paths": [r["image_path"]],
            "image_path": r["image_path"],
            "num_views_used": 1,
            "raw_report_text": r["report"],
            "source_row_index": idx,
        }
        for idx, r in source_df.iterrows()
    ]

# Consolidate and keep only valid image-backed records.
study_df = pd.DataFrame(study_records)
if study_df.empty:
    raise ValueError("No usable study-level records were built from the Kaggle MIMIC-CXR files.")

study_df = study_df.drop_duplicates(subset=[col for col in ["subject_id", "study_id", "image_path", "report_text"] if col in study_df.columns])
study_df["exists"] = study_df["image_path"].map(os.path.exists)
study_df = study_df[study_df["exists"]].copy()

# Build a strict target report with the exact Findings / Impression schema for fine-tuning.
study_df["target_report"] = study_df["report_text"].map(build_structured_report)

# Hackathon-friendly subset keeps the training loop tractable.
subset_df = study_df.sample(n=min(SUBSET_SIZE, len(study_df)), random_state=SEED).reset_index(drop=True)

print(f"Usable study-level rows: {len(study_df):,}")
print(f"Training subset rows:    {len(subset_df):,}")

display_cols = [c for c in ["subject_id", "study_id", "image_path", "num_views_used", "findings", "impression"] if c in subset_df.columns]
display(subset_df[display_cols].head(3))

## Phase 2: 4-Bit Model Loading & LoRA Configuration

In [ ]:
# Code Cell 3: Load PaliGemma in 4-bit + initialize processor (Kaggle T4-safe)

import os
import torch
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration, BitsAndBytesConfig

# Use the model variant you validated.
MODEL_ID = "google/paligemma-3b-mix-224"

# Token is optional if you already logged in with huggingface_hub.login(...).
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")


def _cuda_preflight_message() -> str | None:
    if not torch.cuda.is_available():
        return "CUDA is not available in this runtime. Enable GPU in Kaggle settings."
    major, minor = torch.cuda.get_device_capability(0)
    if major < 7:
        return (
            f"Detected GPU {torch.cuda.get_device_name(0)} with capability sm_{major}{minor}, "
            "but this notebook requires a PyTorch build with sm_70+ support. "
            "Use T4/V100/A100 runtime."
        )
    return None


cuda_issue = _cuda_preflight_message()
if cuda_issue:
    raise RuntimeError(cuda_issue)

# T4 is sm_75, so fp16 is the safe default. bfloat16 is used on newer GPUs (Ampere+).
compute_dtype = (
    torch.bfloat16
    if torch.cuda.get_device_capability(0)[0] >= 8
    else torch.float16
)

# Keep 4-bit loading because downstream Cell 4 uses k-bit preparation + LoRA.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

try:
    processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
    model = PaliGemmaForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        torch_dtype=compute_dtype,
        device_map="auto",
        token=HF_TOKEN,
    )
except Exception as exc:
    raise RuntimeError(
        "Failed to load model. Confirm Hugging Face access (if gated), token/login status, and Kaggle GPU runtime."
    ) from exc

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

print(f"Loaded model: {MODEL_ID}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Compute dtype: {compute_dtype}")
print(f"Trainable params before LoRA: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# Code Cell 4: Configure LoRA adapters and freeze non-target modules

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for low-bit training stability (casts layer norms and enables gradient checkpoint friendliness).
model = prepare_model_for_kbit_training(model)

# Freeze the vision encoder so adaptation budget goes into multimodal-language behavior.
# Clinical rationale: for report generation, language grounding and cross-modal fusion often need
# more adaptation than low-level visual feature extraction when starting from strong pretrained vision backbones.
for name, param in model.named_parameters():
    if "vision_tower" in name or "vision_model" in name:
        param.requires_grad = False

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

# Optional safety freeze for shared embeddings if memory constrained
# for p in model.language_model.model.embed_tokens.parameters():
#     p.requires_grad = False

model.print_trainable_parameters()

# Extra transparency: list a few trainable tensors
trainable_names = [n for n, p in model.named_parameters() if p.requires_grad]
print("Sample trainable parameter names:")
for n in trainable_names[:20]:
    print(" -", n)

## Phase 3: Multimodal Training Loop

In [ ]:
# Code Cell 5: Custom Dataset + collator for multimodal report tuning

from dataclasses import dataclass
from typing import Dict, List

import torch
from torch.utils.data import Dataset

INSTRUCTION_PROMPT = (
    "You are a radiology assistant. Generate a structured chest X-ray report with exactly two sections: "
    "Findings and Impression."
)

class MimicCXRReportDataset(Dataset):
    """
    Returns multimodal training examples for instruction tuning.

    Why this shaping works:
    - Image provides visual evidence.
    - Instruction prompt standardizes behavior.
    - Target text forces consistent clinical report schema.
    """

    def __init__(self, df: pd.DataFrame, processor, image_size: int = 224, max_length: int = 512):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.image_size = image_size
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        row = self.df.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")
        image = image.resize((self.image_size, self.image_size), Image.Resampling.BICUBIC)

        # We concatenate instruction + target report.
        # For causal LM fine-tuning, labels mirror input_ids (pad tokens masked in collator).
        target_text = row["target_report"]
        full_text = f"{INSTRUCTION_PROMPT}\n{target_text}"

        encoded = self.processor(
            images=image,
            text=full_text,
            padding=False,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

        item = {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "pixel_values": encoded["pixel_values"].squeeze(0),
        }
        return item


@dataclass
class CXRDataCollator:
    processor: AutoProcessor

    def __call__(self, features: List[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
        # Dynamic token padding keeps batches memory-efficient.
        input_ids = [f["input_ids"] for f in features]
        attention_mask = [f["attention_mask"] for f in features]
        pixel_values = torch.stack([f["pixel_values"] for f in features])

        batch_tokens = self.processor.tokenizer.pad(
            {"input_ids": input_ids, "attention_mask": attention_mask},
            return_tensors="pt",
        )

        labels = batch_tokens["input_ids"].clone()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {
            "input_ids": batch_tokens["input_ids"],
            "attention_mask": batch_tokens["attention_mask"],
            "pixel_values": pixel_values,
            "labels": labels,
        }


# Split for train/eval demonstration
train_frac = 0.9
split_idx = int(len(subset_df) * train_frac)
train_df = subset_df.iloc[:split_idx].reset_index(drop=True)
eval_df = subset_df.iloc[split_idx:].reset_index(drop=True)

train_dataset = MimicCXRReportDataset(train_df, processor=processor)
eval_dataset = MimicCXRReportDataset(eval_df, processor=processor)
collator = CXRDataCollator(processor=processor)

print(f"Train samples: {len(train_dataset):,}")
print(f"Eval samples: {len(eval_dataset):,}")

In [ ]:
# Code Cell 6: TrainingArguments for QLoRA fine-tuning

from transformers import TrainingArguments

OUTPUT_DIR = "./paligemma-mimiccxr-qlora"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,  # Effective batch improves stability on limited VRAM.
    learning_rate=2e-5,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=20,
    eval_steps=200,
    save_steps=200,
    evaluation_strategy="steps",
    save_strategy="steps",
    save_total_limit=2,
    fp16=(compute_dtype == torch.float16),
    bf16=(compute_dtype == torch.bfloat16),
    optim="paged_adamw_8bit",  # Memory-efficient optimizer for QLoRA workflows.
    gradient_checkpointing=True,
    dataloader_num_workers=2,
    report_to="none",
    remove_unused_columns=False,
)

print(training_args)

In [ ]:
# Code Cell 7: Trainer initialization and fine-tuning

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset if len(eval_dataset) > 0 else None,
    data_collator=collator,
)

train_result = trainer.train()
print(train_result)

# Save LoRA adapter and processor artifacts.
trainer.model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

print(f"Saved fine-tuned adapter artifacts to: {OUTPUT_DIR}")

## Phase 4: Clinical Inference & Metrics

In [ ]:
# Code Cell 8: Inference function for real MIMIC-CXR JPG

import textwrap

@torch.no_grad()
def generate_cxr_report(image_path: str, max_new_tokens: int = 256) -> str:
    """
    Generate a structured report from a chest X-ray image.

    Clinical note:
    - Keep generation deterministic and concise for safer draft reports.
    - This output is a draft and must be reviewed by a qualified radiologist.
    """
    model.eval()

    image = Image.open(image_path).convert("RGB").resize((224, 224), Image.Resampling.BICUBIC)
    prompt = (
        "You are a radiology assistant. Generate a structured chest X-ray report with exactly two sections: "
        "Findings and Impression."
    )

    inputs = processor(images=image, text=prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=1,
        temperature=1.0,
        repetition_penalty=1.05,
    )

    decoded = processor.tokenizer.decode(generated_ids[0], skip_special_tokens=True)

    # Strip echoed prompt when present.
    if decoded.startswith(prompt):
        decoded = decoded[len(prompt):].strip()

    # Enforce structure if model returns free text.
    if "Findings:" not in decoded:
        decoded = f"Findings: {decoded}\nImpression: Review required."
    elif "Impression:" not in decoded:
        decoded = decoded + "\nImpression: Review required."

    return decoded

# Smoke test candidate (first eval sample)
test_image_path = eval_df.iloc[0]["image_path"] if len(eval_df) > 0 else train_df.iloc[0]["image_path"]
print("Test image:", test_image_path)
print(textwrap.shorten(generate_cxr_report(test_image_path), width=240, placeholder=" ..."))

In [ ]:
# Code Cell 9: Visual result panel (X-ray + ground truth + generated report)

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

def plot_case_comparison(row: pd.Series):
    image = Image.open(row["image_path"]).convert("L")
    generated = generate_cxr_report(row["image_path"])
    gt = row["target_report"]

    fig = plt.figure(figsize=(16, 6))
    gs = GridSpec(1, 2, width_ratios=[1, 1.5], figure=fig)

    ax_img = fig.add_subplot(gs[0, 0])
    ax_txt = fig.add_subplot(gs[0, 1])

    ax_img.imshow(image, cmap="gray")
    ax_img.set_title(f"MIMIC-CXR\nsubject_id={row['subject_id']} study_id={row['study_id']}")
    ax_img.axis("off")

    text_block = (
        "GROUND TRUTH REPORT\n"
        + "-" * 30
        + f"\n{gt}\n\n"
        + "MODEL GENERATED REPORT\n"
        + "-" * 30
        + f"\n{generated}"
    )
    ax_txt.text(0.01, 0.99, text_block, va="top", ha="left", fontsize=10, family="monospace", wrap=True)
    ax_txt.axis("off")

    plt.tight_layout()
    plt.show()

# Visualize one held-out example
demo_row = eval_df.iloc[0] if len(eval_df) > 0 else train_df.iloc[0]
plot_case_comparison(demo_row)

In [ ]:
# Code Cell 10 (Optional): BLEU + ROUGE quantitative evaluation

from tqdm.auto import tqdm
import evaluate

# Load metrics (can take a moment first time)
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")

def evaluate_generation(df: pd.DataFrame, max_samples: int = 100) -> pd.DataFrame:
    sample_df = df.sample(n=min(max_samples, len(df)), random_state=SEED).reset_index(drop=True)

    predictions = []
    references = []

    for _, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
        pred = generate_cxr_report(row["image_path"])
        ref = row["target_report"]
        predictions.append(pred)
        references.append(ref)

    rouge_scores = rouge.compute(predictions=predictions, references=references)

    # BLEU expects tokenized refs as list of list
    bleu_scores = bleu.compute(
        predictions=[p.split() for p in predictions],
        references=[[r.split()] for r in references],
    )

    metrics = {
        "rouge1": rouge_scores.get("rouge1", None),
        "rouge2": rouge_scores.get("rouge2", None),
        "rougeL": rouge_scores.get("rougeL", None),
        "bleu": bleu_scores.get("bleu", None),
        "n_samples": len(sample_df),
    }

    return pd.DataFrame([metrics])

metrics_df = evaluate_generation(eval_df if len(eval_df) > 0 else train_df, max_samples=64)
display(metrics_df)